# Autonomous Teleoperation

This notebook uses the trained model to control the Jetbot autonomously. It leverages the model outputs to compute speed and steering, limiting rapid changes to prevent sudden jerks.
It has been optimized with `torch.no_grad()` and FP16 (`half()`) precision to run at high frequency (aiming near 60 fps).

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F
import cv2
import PIL.Image
import numpy as np
import time

import traitlets
import ipywidgets as widgets
from IPython.display import display

from jetbot import Robot, Camera, bgr8_to_jpeg

### Optimize Model with TensorRT (Optional)
Executing the code below will use `torch2trt` to create an optimized TensorRT engine. This process takes a few minutes but is required to reach high FPS (up to 60 FPS) on the Jetson Nano. 
Note: You only need to run this once or whenever you have a new trained model.

In [ ]:
# use to install torch2trt: 
!git clone https://github.com/NVIDIA-AI-IOT/torch2trt
!cd torch2trt
!python setup.py install

In [ ]:
import os
from torch2trt import torch2trt

# 1. Create a dummy input (matching our 224x224 input size)
data = torch.zeros((1, 3, 224, 224)).cuda().half()

# 2. Convert to TensorRT engine
print("Starting TensorRT conversion... (this may take 2-5 minutes)")
model_trt = torch2trt(model, [data], fp16_mode=True)

# 3. Save the optimized engine
trt_model_path = '../models/best_steering_model_xy_trt.pth'
torch.save(model_trt.state_dict(), trt_model_path)
print(f"TensorRT model saved to {trt_model_path}")

# 4. Swap to the TRT model for inference
model = model_trt

### Load the Model
We load our ResNet18 model and set it up for evaluation mode.
Note: For native PyTorch, it's very difficult to reach 60 FPS. Jetson Nano struggles with the framework overhead (usually yielding ~20-30 FPS for ResNet18).
For 60 FPS, the PyTorch model should ideally be compiled into a TensorRT optimized engine using `torch2trt`. 
If you have a TensorRT model (`best_steering_model_xy_trt.pth`), you should load that using `TRTModule()`. 
For now, we'll try to get the fastest native PyTorch performance using `half()` precision.

In [ ]:
import os
from torch2trt import TRTModule

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = torchvision.models.resnet18(pretrained=False)
model.fc = torch.nn.Linear(512, 2)

# Best model path
model_path = '../models/best_steering_model_xy.pth'
trt_model_path = '../models/best_steering_model_xy_trt.pth'

if os.path.exists(trt_model_path):
    print("Loading optimized TensorRT model...")
    model = TRTModule()
    model.load_state_dict(torch.load(trt_model_path))
else:
    print("Loading standard PyTorch model (native)...")
    model.load_state_dict(torch.load(model_path))
    model = model.to(device).eval().half()

### High-Frequency Preprocessing
We pre-load mean/std constants directly to GPU to avoid CPU-GPU syncs on every frame.

In [ ]:
mean = torch.Tensor([0.485, 0.456, 0.406]).cuda().half()
std = torch.Tensor([0.229, 0.224, 0.225]).cuda().half()

def preprocess(image):
    image = PIL.Image.fromarray(image)
    image = transforms.functional.resize(image, (224, 224))
    image = transforms.functional.to_tensor(image).to(device).half()
    image.sub_(mean[:, None, None]).div_(std[:, None, None])
    return image[None, ...]

### Initialize Camera and Robot
Prepare the camera output and basic widgets so we can monitor what Jetbot sees.

In [ ]:
camera = Camera.instance()
robot = Robot()

image_widget = widgets.Image()
traitlets.dlink((camera, 'value'), (image_widget, 'value'), transform=bgr8_to_jpeg)

display(image_widget)

### UI Controls for Driving and Output Filters
Adjust `max_speed_delta` and `max_steering_delta` to limit how fast speed and direction can change.
This smoothing disallows sharp changes across individual frames.

In [ ]:
speed_gain_slider = widgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.3, description='speed gain')
steering_gain_slider = widgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.4, description='steering gain')
steering_dgain_slider = widgets.FloatSlider(min=0.0, max=0.5, step=0.001, value=0.0, description='steering kd')
steering_bias_slider = widgets.FloatSlider(min=-0.5, max=0.5, step=0.01, value=0.0, description='steering bias')

# Delta clippers to prevent sharp jumps in control signals per frame
max_speed_delta = widgets.FloatSlider(min=0.01, max=0.2, step=0.01, value=0.05, description='max spd $\\Delta$')
max_steer_delta = widgets.FloatSlider(min=0.01, max=0.5, step=0.01, value=0.10, description='max str $\\Delta$')

# Limit FPS
target_fps_slider = widgets.IntSlider(min=1, max=60, step=1, value=15, description='target FPS')

controls_box = widgets.VBox([
    speed_gain_slider, 
    steering_gain_slider, 
    steering_dgain_slider, 
    steering_bias_slider,
    max_speed_delta,
    max_steer_delta,
    target_fps_slider
])
display(controls_box)

x_slider = widgets.FloatSlider(min=-1.0, max=1.0, description='network x')
y_slider = widgets.FloatSlider(min=0, max=1.0, orientation='vertical', description='network y')
steering_slider = widgets.FloatSlider(min=-1.0, max=1.0, description='actual steer')
speed_slider = widgets.FloatSlider(min=0, max=1.0, orientation='vertical', description='actual speed')

display(widgets.HBox([y_slider, speed_slider]))
display(x_slider, steering_slider)

### Inference Cycle
Listens to camera feed changes and directly evaluates to motor control.

In [ ]:
angle = 0.0
angle_last = 0.0

current_speed = 0.0
current_steering = 0.0
last_inference_time = 0.0

def clamp_delta(new_val, old_val, max_delta):
    """Restricts how rapidly a value can change compared to the previous frame."""
    if new_val - old_val > max_delta:
        return old_val + max_delta
    elif old_val - new_val > max_delta:
        return old_val - max_delta
    return new_val

def execute(change):
    global angle, angle_last
    global current_speed, current_steering, last_inference_time
    
    current_time = time.time()
    # Enforce FPS limit
    if current_time - last_inference_time < 1.0 / target_fps_slider.value:
        return
    last_inference_time = current_time
    
    image = change['new']
    
    # Inference mode skips gradient tracking for higher FPS
    with torch.no_grad():
        xy = model(preprocess(image)).detach().float().cpu().numpy().flatten()
        
    x = xy[0]
    y = (0.5 - xy[1]) / 2.0
    
    x_slider.value = x
    y_slider.value = y
    
    # Proportional-Derivative calculation from NVIDIA example
    angle = np.arctan2(x, y)
    pid = angle * steering_gain_slider.value + (angle - angle_last) * steering_dgain_slider.value
    angle_last = angle
    
    target_steering = pid + steering_bias_slider.value
    target_speed = speed_gain_slider.value
    
    # --- Constraints: Clip change rate per frame ---
    max_sd = max_speed_delta.value
    max_td = max_steer_delta.value
    
    current_speed = clamp_delta(target_speed, current_speed, max_sd)
    current_steering = clamp_delta(target_steering, current_steering, max_td)
    
    # Drive the widgets to visualize output signal
    steering_slider.value = current_steering
    speed_slider.value = current_speed
    
    # Convert to motor power levels
    left_power = max(min(current_speed + current_steering, 1.0), 0.0)
    right_power = max(min(current_speed - current_steering, 1.0), 0.0)
    
    robot.left_motor.value = left_power
    robot.right_motor.value = right_power

### **Start driving!**
Executes whenever the camera captures a fresh frame.

In [ ]:
camera.observe(execute, names='value')

### Stop
Detach observation and pause robot.

In [ ]:
camera.unobserve(execute, names='value')
time.sleep(0.1)
robot.stop()